# Andere WebArena-Verified Sites mit BrowserGym testen

Nach GitLab Task 44 testen wir hier eine zweite Site. Ziel ist nicht sofort, eine Shopping-/Reddit-Aufgabe zu loesen, sondern kontrolliert zu pruefen:

1. offizielle WebArena-Verified Environment-CLI kann eine Site starten,
2. Config passt zur Site,
3. `agent-input-get` rendert echte URLs,
4. BrowserGym kann die Startseite oeffnen und HAR/Observation erzeugen.

Erst danach kommt ein echter Agent fuer diese Site.

In [ ]:
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OFFICIAL_REPO = PROJECT_ROOT / 'external' / 'webarena-verified'
OFFICIAL_REPO

## 1. Site auswaehlen

`shopping`, `shopping_admin` und `reddit` sind gute naechste Smoke-Tests. `wikipedia` und `map` brauchen vorher Daten-Setup und kommen spaeter.

In [ ]:
SITE = 'shopping'
SITE_ENV = {
    'shopping': ('__SHOPPING__', 'http://localhost:7770', {'username': 'emma.lopez@gmail.com', 'password': 'Password.123'}),
    'shopping_admin': ('__SHOPPING_ADMIN__', 'http://localhost:7780', {'username': 'admin', 'password': 'admin1234'}),
    'reddit': ('__REDDIT__', 'http://localhost:9999', {'username': 'MarvelsGrantMan136', 'password': 'test1234'}),
}
assert SITE in SITE_ENV
SITE_ENV[SITE]

## 2. Environment starten

Diese Zelle startet den offiziellen WebArena-Verified-Container fuer die Site. Das kann beim ersten Mal ein Docker-Image ziehen.

In [ ]:
RUN_START_SITE = False

if RUN_START_SITE:
    subprocess.run(['uv', 'run', 'webarena-verified', 'env', 'start', '--site', SITE], cwd=OFFICIAL_REPO, check=True)
else:
    print('Start uebersprungen. Setze RUN_START_SITE = True oder starte im Terminal:')
    print(f'cd {OFFICIAL_REPO}')
    print(f'uv run webarena-verified env start --site {SITE}')

In [ ]:
status = subprocess.run(
    ['uv', 'run', 'webarena-verified', 'env', 'status', '--site', SITE],
    cwd=OFFICIAL_REPO,
    text=True,
    capture_output=True,
)
print(status.stdout)
print(status.stderr)
if status.returncode != 0:
    raise RuntimeError(
        f'Die Site {SITE!r} laeuft noch nicht. Setze oben RUN_START_SITE = True '
        f'oder starte im Terminal: cd {OFFICIAL_REPO} && uv run webarena-verified env start --site {SITE}'
    )
print(f'{SITE} ist laut WebArena-Verified env status bereit.')

## 3. Lokale Config fuer diese Site schreiben

In [ ]:
env_key, base_url, credentials = SITE_ENV[SITE]
config = {'environments': {env_key: {'urls': [base_url], 'credentials': credentials}}}
config_path = OFFICIAL_REPO / 'output' / f'config.{SITE}.local.json'
config_path.parent.mkdir(exist_ok=True)
config_path.write_text(json.dumps(config, indent=2))
config_path, config

## 4. Kandidaten-Tasks exportieren

Wir nehmen zuerst `NAVIGATE`-Tasks, weil sie der GitLab-Aufgabe 44 am aehnlichsten sind. Nicht jede Site hat solche Tasks.

In [ ]:
candidates_path = OFFICIAL_REPO / 'output' / f'{SITE}_navigate_tasks.json'
proc = subprocess.run([
    'uv', 'run', 'webarena-verified', 'dataset-get',
    '--sites', SITE,
    '--task-type', 'NAVIGATE',
    '--fields', 'task_id,sites,intent,intent_template_id',
    '--output', str(candidates_path.relative_to(OFFICIAL_REPO)),
], cwd=OFFICIAL_REPO, text=True, capture_output=True)
print(proc.stdout)
print(proc.stderr)
if proc.returncode != 0:
    print('Keine NAVIGATE-Tasks fuer diese Site gefunden. Fuer Reddit z. B. spaeter RETRIEVE/MUTATE testen.')
else:
    candidates = json.loads(candidates_path.read_text())
    print('Anzahl Kandidaten:', len(candidates))
    candidates[:10]

## 5. Agent-Input fuer einen Kandidaten rendern

In [ ]:
TASK_ID = candidates[0]['task_id']
tasks_file = OFFICIAL_REPO / 'output' / f'{SITE}_task_{TASK_ID}.json'
subprocess.run([
    'uv', 'run', 'webarena-verified', 'agent-input-get',
    '--task-ids', str(TASK_ID),
    '--config', str(config_path.relative_to(OFFICIAL_REPO)),
    '--output', str(tasks_file.relative_to(OFFICIAL_REPO)),
], cwd=OFFICIAL_REPO, check=True)

task_input = json.loads(tasks_file.read_text())
task_input

## 6. BrowserGym Site-Probe

Dieser Probe oeffnet nur die Start-URL mit BrowserGym und schreibt HAR + Metadaten. Er behauptet nicht, die Aufgabe geloest zu haben.

Wenn hier `Connection refused` kommt, ist fast immer die Site-Umgebung nicht gestartet oder noch nicht bereit. Dann zuerst die Status-Zelle oben reparieren.

In [ ]:
subprocess.run([
    str(PROJECT_ROOT / '.venv/bin/python'),
    str(PROJECT_ROOT / 'scripts/run_browsergym_site_probe.py'),
    '--tasks-file', str(tasks_file),
    '--task-id', str(TASK_ID),
    '--output-root', str(OFFICIAL_REPO / 'output/site-probe'),
], cwd=PROJECT_ROOT, check=True)

## 7. Probe-Artefakte ansehen

In [ ]:
probe_dir = OFFICIAL_REPO / 'output' / 'site-probe' / SITE / str(TASK_ID)
sorted(p.name for p in probe_dir.iterdir())

In [ ]:
json.loads((probe_dir / 'probe_metadata.json').read_text())

## Einordnung

Wenn diese Probe funktioniert, ist die naechste Site technisch erreichbar und BrowserGym kann sie oeffnen. Das ist noch keine Benchmark-Loesung. Der naechste echte Schritt waere ein kleiner scripted Agent fuer genau einen einfachen Task auf dieser Site oder danach AgentLab als Experiment-Rahmen.